# Lab qc_05 — Variational quantum eigensolver with parameter-shift gradients

**Skill:** `qc.variational` · ~75 min · CPU

### Goals
1. Encode $H = Z\otimes Z + 0.5\,X\otimes I$ as a matrix and as a Qiskit `SparsePauliOp`, and get the exact ground energy with `np.linalg.eigvalsh`.
2. Write a 4-parameter ansatz, the energy $E(\boldsymbol\theta) = \langle\psi(\boldsymbol\theta)\vert H\vert\psi(\boldsymbol\theta)\rangle$ and its **exact** gradient via the parameter-shift rule.
3. Run gradient descent to the ground energy.

### Theory recap
**Variational principle:** $E(\boldsymbol\theta) \ge E_0$ for every normalised state, with equality at the ground state.

**Parameter shift** (Mitarai et al. 2018; Schuld et al. 2019). If $\theta$ enters only through $e^{-i\theta P/2}$ with $P^2 = I$ (e.g. $R_y$), then $E$ is a sinusoid in $\theta$ with period $2\pi$ and
$$\frac{\partial E}{\partial\theta_i} = \frac{1}{2}\Big[E(\boldsymbol\theta + \tfrac{\pi}{2}\mathbf e_i) - E(\boldsymbol\theta - \tfrac{\pi}{2}\mathbf e_i)\Big]$$
— exact, with two circuit evaluations per parameter. Example: $\langle Z\rangle = \cos\theta$ for $R_y(\theta)\lvert0\rangle$, so $\partial_\theta\langle Z\rangle = -\sin\theta$.

**Ordering trap.** Qiskit Pauli labels are little-endian: `"XI"` means $X$ on **qubit 1** and $I$ on qubit 0, i.e. the matrix `np.kron(X, I)` in Qiskit's amplitude order.
Paper answer to check against: in each $Z_0$ sector, $H = \pm Z_1 + 0.5X_1$, so $E_0 = -\sqrt{1 + 0.25}$.

### Read
- IBM Quantum Learning, *Variational Algorithm Design* — https://quantum.cloud.ibm.com/learning/en/courses/variational-algorithm-design
- Qiskit `SparsePauliOp` / `Statevector.expectation_value` — https://github.com/Qiskit/qiskit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

HAMILTONIAN = SparsePauliOp.from_list([("ZZ", 1.0), ("XI", 0.5)])
N_PARAMS = 4

## 1. Hamiltonian and exact answer

In [ ]:
def hamiltonian_matrix():
    # TODO: Z⊗Z + 0.5 X⊗I as a 4x4 numpy array in Qiskit's ordering
    raise NotImplementedError

def exact_ground_energy():
    # TODO: smallest eigenvalue (np.linalg.eigvalsh)
    raise NotImplementedError

## 2. Ansatz and energy
`ansatz(params)`: `ry(p0, 0)`, `ry(p1, 1)`, `cx(0, 1)`, `ry(p2, 0)`, `ry(p3, 1)`. `energy(params)` via `Statevector(...).expectation_value(HAMILTONIAN)` (return a real float).

In [ ]:
def ansatz(params):
    # TODO
    raise NotImplementedError

def energy(params):
    # TODO
    raise NotImplementedError

## 3. Gradients
`parameter_shift_grad(params)` → numpy array of length 4. Also `single_qubit_expz_grad(theta)`: the parameter-shift derivative of $f(\theta) = \cos\theta$ (should equal $-\sin\theta$).

In [ ]:
def parameter_shift_grad(params):
    # TODO
    raise NotImplementedError

def single_qubit_expz_grad(theta):
    # TODO
    raise NotImplementedError

## 4. Optimise
`vqe(init_params, lr, steps)` → `(params, history)` with `history[0] = energy(init)` and one entry per step.

In [ ]:
def vqe(init_params, lr=0.2, steps=200):
    # TODO: plain gradient descent with parameter-shift gradients
    raise NotImplementedError

In [ ]:
p, hist = vqe(np.array([0.1, 0.2, -0.3, 0.4]), lr=0.2, steps=150)
e0 = exact_ground_energy()
plt.semilogy(np.array(hist) - e0 + 1e-12)
plt.xlabel("step"); plt.ylabel("E - E0"); plt.title("VQE convergence"); plt.show()
print(f"VQE: {hist[-1]:.6f}   exact: {e0:.6f}")

# energy landscape along one parameter, with the parameter-shift gradient at a point
th = np.linspace(-np.pi, np.pi, 200)
E1 = [energy([t, 0, 0, 0]) for t in th]
plt.plot(th, E1); plt.xlabel("θ0"); plt.ylabel("E"); plt.title("a sinusoid in each parameter"); plt.show()

### Reflect (on paper)
1. Prove the parameter-shift rule for $R_y$: write $E(\theta) = a + b\cos\theta + c\sin\theta$.
2. With $S$ shots per evaluation, what is the variance of the parameter-shift estimate? Why do finite differences with small $h$ fare worse on hardware?
3. QAOA for MaxCut on one edge uses $H_C = \tfrac12(1 - Z\otimes Z)$; what is the depth-1 optimum?

## Grade
Run all cells above, then this one. It runs the hidden checks and records the result in the Learning Center.

In [ ]:
from lc_client import grade
grade("qc_05_vqe", globals())